System construction and test


In [33]:
from datetime import date, datetime
import pandas as pd
#import yfinance as yf
import time
import numpy as np
pd.options.mode.chained_assignment = None  # default='warn'
from itertools import product
import sqlite3


In [35]:
id_backtest = 4

In [37]:
import sqlite3
import pandas as pd

# Conecta ao banco de dados SQLite
con = sqlite3.connect('sqTradeSys.db')  # ou o caminho correto do seu arquivo .sqlite

# Consulta SQL para extrair o registro
query = f"SELECT * FROM vwbacktest WHERE id_backtest = {id_backtest}"

# Executa a consulta e lê em um DataFrame
df = pd.read_sql_query(query, con)

# Converte o primeiro (e único) registro em Series
srbacktest = df.iloc[0] if not df.empty else None

# Fecha a conexão (opcional)
con.close()
display (srbacktest)

id_backtest                                                    4
id_titulos                                                     3
dataini                                      2023-10-10 22:00:00
datafim                                      2025-10-03 21:00:00
symbol                                                   BTC-USD
intervalo                                                   1dia
moeda                                                        USD
source         C:\Users\scitr\anaconda_projects\Trading_Syste...
Name: 0, dtype: object

In [39]:
dataini = srbacktest['dataini']
datafim =  srbacktest['datafim']
display (dataini , datafim)

'2023-10-10 22:00:00'

'2025-10-03 21:00:00'

In [41]:
#%%timeit
import sqlite3
import pandas as pd

# Caminho para o banco de dados

caminho_bd =  srbacktest['source'] 
# Conectando ao banco
conexao = sqlite3.connect(caminho_bd)

# Lendo a view
#consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
consulta = f"""
SELECT * FROM vwtitulosdados
WHERE datetime BETWEEN '{dataini}' AND '{datafim}' AND symbol = '{srbacktest['symbol']}' AND intervalo = '{srbacktest['intervalo']}' AND moeda = '{srbacktest['moeda']}'
ORDER BY datetime
"""

dftitulosdados = pd.read_sql_query(consulta, conexao)

# Fechando a conexão
conexao.close()

# Exibindo os primeiros registros para conferir
#display(dftitulosdados)
dftitulosdados = dftitulosdados.drop(columns=["symbol", "moeda", "intervalo"])
display(len(dftitulosdados))
display(dftitulosdados.head(10))

724

,datetime,open,high,low,close,volume
0,2023-10-11 00:00:00,27392.076172,27474.115234,26561.099609,26873.320312,1.364809e+10
1,2023-10-12 00:00:00,26873.292969,26921.439453,26558.320312,26756.798828,9.392909e+09
2,2023-10-13 00:00:00,26752.878906,27092.697266,26686.322266,26862.375000,1.516531e+10
3,2023-10-14 00:00:00,26866.203125,26969.000000,26814.585938,26861.707031,5.388117e+09
4,2023-10-15 00:00:00,26858.011719,27289.169922,26817.894531,27159.652344,7.098202e+09
5,2023-10-16 00:00:00,27162.628906,29448.138672,27130.472656,28519.466797,2.783388e+10
6,2023-10-17 00:00:00,28522.097656,28618.751953,28110.185547,28415.748047,1.487253e+10
7,2023-10-18 00:00:00,28413.531250,28889.009766,28174.251953,28328.341797,1.272413e+10
8,2023-10-19 00:00:00,28332.416016,28892.474609,28177.988281,28719.806641,1.444806e+10
9,2023-10-20 00:00:00,28732.812500,30104.085938,28601.669922,29682.949219,2.153613e+10


In [43]:
# Conecta ao banco de dados SQLite
con = sqlite3.connect('sqTradeSys.db')  # ou o caminho correto do seu arquivo .sqlite

# Consulta SQL para extrair o registro
#query = f"SELECT * FROM vwbacktestparameters ,name, max, min,step WHERE id_backtest = {id_backtest}"
query = f"SELECT  name, type, max, min, step FROM vwbacktestparameters WHERE id_backtest = {id_backtest}"
# Executa a consulta e lê em um DataFrame
df = pd.read_sql_query(query, con)

# Fecha a conexão (opcional)
con.close()

display (df)

,name,type,max,min,step
0,K,int,15,5,5
1,D,int,15,5,5
2,smoth,int,5,5,5
3,medM,int,4,4,1
4,lowM,int,4,4,1
5,stpl,float,0.02,0.02,0.01
6,comission,float,0.003,0.003,0.003
7,drawmax,float,0.2,0.2,0.1


In [45]:
import pandas as pd
import numpy as np
from itertools import product

def parameters_combinator(dfcomb: pd.DataFrame) -> pd.DataFrame:
    """
    Gera um DataFrame com todas as combinações possíveis de parâmetros
    definidos em dfcomb, respeitando os tipos especificados.

    Parâmetros esperados em dfcomb:
    - name: nome da coluna
    - type: tipo de dado ('int' ou 'float')
    - min: valor mínimo
    - max: valor máximo
    - step: incremento

    Retorna:
    - dfparamtest: DataFrame com todas as combinações possíveis
    """
    param_ranges = {}

    for _, row in dfcomb.iterrows():
        name = row['name']
        tipo = row['type']

        # Converte min, max, step para o tipo correto
        if tipo == 'int':
            min_val = int(row['min'])
            max_val = int(row['max'])
            step_val = int(row['step'])
        elif tipo == 'float':
            min_val = float(row['min'])
            max_val = float(row['max'])
            step_val = float(row['step'])
        else:
            raise ValueError(f"Tipo não suportado: {tipo}")

        # Gera a faixa de valores
        values = np.round(np.arange(min_val, max_val + step_val, step_val), 5)
        param_ranges[name] = values

    # Gera todas as combinações possíveis
    combinations = list(product(*param_ranges.values()))

    # Cria o novo DataFrame
    dfparamtest = pd.DataFrame(combinations, columns=param_ranges.keys())

    # Aplica os tipos definidos
    for _, row in dfcomb.iterrows():
        col = row['name']
        tipo = row['type']
        if tipo == 'int':
            dfparamtest[col] = dfparamtest[col].astype(int)
        elif tipo == 'float':
            dfparamtest[col] = dfparamtest[col].astype(float)

    return dfparamtest
dfcomb = df

In [100]:
dfcomb = df
dfparamtest = parameters_combinator(dfcomb)
print(dfparamtest)
print(len(dfparamtest))

     K   D  smoth  medM  lowM  stpl  comission  drawmax
0    5   5      5     4     4  0.02      0.003      0.2
1    5   5      5     4     4  0.02      0.003      0.3
2    5  10      5     4     4  0.02      0.003      0.2
3    5  10      5     4     4  0.02      0.003      0.3
4    5  15      5     4     4  0.02      0.003      0.2
5    5  15      5     4     4  0.02      0.003      0.3
6   10   5      5     4     4  0.02      0.003      0.2
7   10   5      5     4     4  0.02      0.003      0.3
8   10  10      5     4     4  0.02      0.003      0.2
9   10  10      5     4     4  0.02      0.003      0.3
10  10  15      5     4     4  0.02      0.003      0.2
11  10  15      5     4     4  0.02      0.003      0.3
12  15   5      5     4     4  0.02      0.003      0.2
13  15   5      5     4     4  0.02      0.003      0.3
14  15  10      5     4     4  0.02      0.003      0.2
15  15  10      5     4     4  0.02      0.003      0.3
16  15  15      5     4     4  0.02      0.003  

In [288]:
dfmetricas = None

START LOOP

In [467]:
il = 12  #16 ,0,4,8

In [469]:
# Parameters

# System parameters
# stoch_hml_1
K = dfparamtest.loc[il,'K']
D = dfparamtest.loc[il,'D']
smoth = dfparamtest.loc[il,'smoth']
medM = (dfparamtest.loc[il,'medM'])
lowM = (dfparamtest.loc[il,'lowM'])

# Backtesting parameters

stpl = dfparamtest.loc[il,'stpl']
comission = dfparamtest.loc[il,'comission']
drawmax = dfparamtest.loc[il,'drawmax']
print(K,D, smoth, stpl, drawmax)

15 5 5 0.02 0.2


In [471]:


lsmetricas = []
srparamtest = dfparamtest.loc[il]
lsmetricas.append(srparamtest)

print(srparamtest)


K            15.000
D             5.000
smoth         5.000
medM          4.000
lowM          4.000
stpl          0.020
comission     0.003
drawmax       0.200
Name: 12, dtype: float64


Trading System

In [474]:
# Stochastic calculation
def stochastic(dftitulosdados, i, K, D, smoth):
    df = dftitulosdados.copy()
    df["k"] = (100. * (df.close - df.low.rolling(K).min()) /
               (df.high.rolling(K).max() - df.low.rolling(K).min()))
    
    df["k" + i] = df["k"].rolling(smoth).mean()
    df["d" + i] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)
    return df

dfstoch = stochastic(dftitulosdados, 'high', K, D, smoth)
display (dfstoch.tail(10))

,datetime,open,high,low,close,volume,khigh,dhigh
714,2025-09-24 00:00:00,112007.664062,113986.273438,111229.640625,113328.632812,4.804460e+10,43.818792,64.749546
715,2025-09-25 00:00:00,113330.164062,113541.085938,108713.398438,109049.289062,7.552865e+10,30.101628,53.876298
716,2025-09-26 00:00:00,109041.296875,110359.195312,108728.976562,109712.828125,5.773829e+10,19.042577,42.069001
717,2025-09-27 00:00:00,109707.140625,109778.500000,109144.296875,109681.945312,2.630804e+10,15.331023,31.924847
718,2025-09-28 00:00:00,109681.945312,112375.484375,109236.945312,112122.640625,3.337105e+10,19.274109,25.513626
719,2025-09-29 00:00:00,112117.875000,114473.570312,111589.953125,114400.382812,6.000015e+10,24.787149,21.707297
720,2025-09-30 00:00:00,114396.523438,114836.617188,112740.562500,114056.085938,5.898633e+10,35.673394,22.821650
721,2025-10-01 00:00:00,114057.593750,118648.929688,113981.398438,118648.929688,7.132868e+10,53.500341,29.713203
722,2025-10-02 00:00:00,118652.382812,121086.406250,118383.156250,120681.257812,7.141516e+10,70.739545,40.794907
723,2025-10-03 00:00:00,120606.320312,123850.195312,119392.710938,122444.242188,8.526993e+10,81.469189,53.233923


In [476]:
# Stochastic high, med and low frequency
def stoch_hml(dfstoch, k, d, smth, medM, lowM):
    df = dfstoch.copy()
    df = stochastic(df, "high", k, d, smth)
    df = stochastic(df, "med", k * medM, d * medM, smth * medM)
    df = stochastic(df, "low", k * medM * lowM, d * medM * lowM, smth * medM * lowM)
    return df
pd.set_option('display.max_rows', None)
dfstoch_hml= stoch_hml(dfstoch , K, D, smoth, medM , lowM )
display(dfstoch_hml.tail(10))

,datetime,open,high,low,close,volume,khigh,dhigh,kmed,dmed,klow,dlow
714,2025-09-24 00:00:00,112007.664062,113986.273438,111229.640625,113328.632812,4.804460e+10,43.818792,64.749546,40.104489,32.746282,83.794983,87.487966
715,2025-09-25 00:00:00,113330.164062,113541.085938,108713.398438,109049.289062,7.552865e+10,30.101628,53.876298,39.638499,32.881775,83.532180,87.466810
716,2025-09-26 00:00:00,109041.296875,110359.195312,108728.976562,109712.828125,5.773829e+10,19.042577,42.069001,39.489579,33.149952,83.264297,87.436912
717,2025-09-27 00:00:00,109707.140625,109778.500000,109144.296875,109681.945312,2.630804e+10,15.331023,31.924847,39.057344,33.511459,82.916503,87.396317
718,2025-09-28 00:00:00,109681.945312,112375.484375,109236.945312,112122.640625,3.337105e+10,19.274109,25.513626,39.072243,33.942824,82.626691,87.346358
719,2025-09-29 00:00:00,112117.875000,114473.570312,111589.953125,114400.382812,6.000015e+10,24.787149,21.707297,39.907180,34.506709,82.413069,87.289867
720,2025-09-30 00:00:00,114396.523438,114836.617188,112740.562500,114056.085938,5.898633e+10,35.673394,22.821650,39.936485,35.103236,82.193144,87.226865
721,2025-10-01 00:00:00,114057.593750,118648.929688,113981.398438,118648.929688,7.132868e+10,53.500341,29.713203,40.850427,35.779498,82.057259,87.158086
722,2025-10-02 00:00:00,118652.382812,121086.406250,118383.156250,120681.257812,7.141516e+10,70.739545,40.794907,42.182819,36.526271,82.046189,87.086230
723,2025-10-03 00:00:00,120606.320312,123850.195312,119392.710938,122444.242188,8.526993e+10,81.469189,53.233923,44.072077,37.348779,82.132421,87.013159


In [477]:
# Trading system: Stoch_HighMedLow_Long






# Criteria calculation
def system_criterias(dfstoch_hml):
    df = dfstoch_hml.copy()
    df["longbuylow"] = ((df["klow"] > 20) & (df["klow"] > df["dlow"])).astype(int)
    df["longbuymed"] = ((df["kmed"] > 20) & (df["kmed"] > df["dmed"])).astype(int)
    df["longbuyhigh"] = ((df["khigh"] > 20) & (df["khigh"] > df["dhigh"])).astype(int)
    return df

# Signal generation
def system_signals(dfcriterias):
    df = dfcriterias.copy()
    n = len(df)
    state_array = np.full(n, "standby", dtype=object)
    estado_anterior = "standby"

    high = df["longbuyhigh"].to_numpy()
    med = df["longbuymed"].to_numpy()
    low = df["longbuylow"].to_numpy()

    for i in range(1, n):
        if high[i] == 1 and med[i] == 1 and low[i] == 1 and estado_anterior == "standby":
            state_array[i] = "enter"
            estado_anterior = "enter"
        elif med[i] == 1 and low[i] == 1 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "stay"
            estado_anterior = "stay"
        elif high[i] == 1 and low[i] == 1 and med[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "stay"
            estado_anterior = "stay"
        elif low[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "out"
            estado_anterior = "out"
        elif high[i] == 0 and med[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "out"
            estado_anterior = "out"
        elif (low[i] == 0 or med[i] == 0) and estado_anterior == "out":
            state_array[i] = "standby"
            estado_anterior = "standby"
        else:
            state_array[i] = estado_anterior

    df["state"] = state_array
    dfsignals = df.drop(columns=[
        "khigh", "dhigh", "kmed", "dmed", "klow", "dlow",
        "longbuylow", "longbuymed", "longbuyhigh"
    ])
    return dfsignals

def Stoch_HighMedLow_Long (dftitulosdados, K, D, smoth, medM , lowM):     #Columns: datetime,open,high,low,close,volume 
    dfstoch = stochastic(dftitulosdados, 'high', K, D, smoth)
    
    dfcriterias = system_criterias (dfstoch_hml)
    dfsignals = system_signals (dfcriterias)
    return dfsignals                                                      #Columns: datetime,open,high,low,close,volume,state

In [480]:
#%%timeit
pd.set_option('display.max_rows', None)

dfsignals = Stoch_HighMedLow_Long (dftitulosdados, K, D, smoth, medM , lowM)
display(dfsignals)

,datetime,open,high,low,close,volume,state
0,2023-10-11 00:00:00,27392.076172,27474.115234,26561.099609,26873.320312,1.364809e+10,standby
1,2023-10-12 00:00:00,26873.292969,26921.439453,26558.320312,26756.798828,9.392909e+09,standby
2,2023-10-13 00:00:00,26752.878906,27092.697266,26686.322266,26862.375000,1.516531e+10,standby
3,2023-10-14 00:00:00,26866.203125,26969.000000,26814.585938,26861.707031,5.388117e+09,standby
4,2023-10-15 00:00:00,26858.011719,27289.169922,26817.894531,27159.652344,7.098202e+09,standby
5,2023-10-16 00:00:00,27162.628906,29448.138672,27130.472656,28519.466797,2.783388e+10,standby
6,2023-10-17 00:00:00,28522.097656,28618.751953,28110.185547,28415.748047,1.487253e+10,standby
7,2023-10-18 00:00:00,28413.531250,28889.009766,28174.251953,28328.341797,1.272413e+10,standby
8,2023-10-19 00:00:00,28332.416016,28892.474609,28177.988281,28719.806641,1.444806e+10,standby
9,2023-10-20 00:00:00,28732.812500,30104.085938,28601.669922,29682.949219,2.153613e+10,standby


In [481]:
                                 #stop_loss_reentry

def stop_loss_reentry (dfsignals, stpl) :                 #Columns: datetime,open,high,low,close,volume,state

    df = dfsignals[(dfsignals['state'] == 'enter') | (dfsignals['state'] == 'stay')]
    df = df.reset_index(drop=True)
    
    
    df["stpl"] = 0.0
    stoplossprice = 0.0
    lastlongbuyprice = 0.0

    for i in range(0, len(df)):
        
        if df.loc[i, "state"] == "enter" :
           stoplossprice = df.loc[i, "close"]
           df.loc[i,"stpl"] = df.loc[i, "close"] - stoplossprice * (1 - stpl)
           
        if df.loc[i, "state"]== "stay" :
           df.loc[i, "stpl"] = df.loc[i, "close"] - stoplossprice * (1- stpl)
            
           if df.loc[i, "stpl"] < 0.0 :
                df.loc[i, "state"] = "out"
               
           if df.loc[i, "stpl"] > 0.0 and   (df.loc[i-1, "state"] == "out" or df.loc[i-1, "state"] == "outstpl") :
                df.loc[i, "state"] = "enter"
               
           if df.loc[i, "stpl"] < 0.0 and   (df.loc[i-1, "state"] == "out" or df.loc[i-1, "state"] == "outstpl") :
                df.loc[i, "state"] = "outstpl"
    dfstoploss = df
    return dfstoploss                               #Columns: datetime,open,high,low,close,volume,state,stpl


In [482]:
dfstoploss = stop_loss_reentry (dfsignals, stpl)
display (dfstoploss)


,datetime,open,high,low,close,volume,state,stpl
0,2024-11-11 00:00:00,80471.414062,89604.500000,80283.250000,88701.484375,1.179668e+11,enter,1774.029688
1,2024-11-12 00:00:00,88705.562500,89956.882812,85155.109375,87955.812500,1.336733e+11,stay,1028.357813
2,2024-11-13 00:00:00,87929.968750,93434.351562,86256.929688,90584.164062,1.235590e+11,stay,3656.709375
3,2024-11-14 00:00:00,90574.882812,91765.218750,86682.812500,87250.429688,8.761671e+10,stay,322.975000
4,2024-11-15 00:00:00,87284.179688,91868.742188,87124.898438,91066.007812,7.824311e+10,stay,4138.553125
5,2024-11-16 00:00:00,91064.367188,91763.945312,90094.226562,90558.476562,4.433319e+10,stay,3631.021875
6,2024-11-17 00:00:00,90558.460938,91433.039062,88741.664062,89845.851562,4.635016e+10,stay,2918.396875
7,2024-11-18 00:00:00,89843.718750,92596.789062,89393.593750,90542.640625,7.553578e+10,stay,3615.185938
8,2024-11-19 00:00:00,90536.812500,94002.867188,90426.984375,92343.789062,7.452105e+10,stay,5416.334375
9,2024-11-20 00:00:00,92341.890625,94902.023438,91619.500000,94339.492188,7.173096e+10,stay,7412.037500


In [483]:
#Index_sc, Index, Trade calculation

# clean dfsignals "out" duplicates for StopLoss
def index_dataframe (dfsignals, dfstoploss=None) :
    
    if dfstoploss is None or dfstoploss.empty:   #em caso de no aplicar ningum stoploss
        
       dfsignalsenterout = dfsignals[(dfsignals['state'] == 'enter') | (dfsignals['state'] == 'out')]
    else :
        dfsignalsout = dfsignals[(dfsignals['state'] == 'out')]
        # elimino a culuna stpl de dfstoploss e filtro os valores enter e out 
        dfstoplossdrop = dfstoploss.drop(columns=["stpl"]) 
        dfstoplossenterout = dfstoplossdrop[(dfstoplossdrop['state'] == 'enter') | (dfstoplossdrop['state'] == 'out')]

        # concatenar os dois df para ter o total dos signals enter e out
        dfsignalsenterout = pd.concat([dfsignalsout, dfstoplossenterout], ignore_index=True)     
       
    

    # Ordenar pelo datetime e resetear o index
    dfsignalsenterout["datetime"] = pd.to_datetime(dfsignalsenterout["datetime"])
    dfsignalsenterout = dfsignalsenterout.sort_values("datetime").reset_index(drop=True)

    # limpar os out duplicados"out" por a saida anticipada do stoploss e reiniciar indice
    df = dfsignalsenterout
    cond = (df["state"] == "out")  & (df["state"].shift(1) == "out")
    dfsignalsentoutclean = df[~cond].reset_index(drop=True)
    return dfsignalsentoutclean 



# Indexes calculation
def index_trade(dfsignalsentoutclean):
    df = dfsignalsentoutclean
    df ["index_sc"] = 100. 
    df ["trade"] = 0.
    df ["index"] = 100. *(1-comission) 
    for i in range(1, len(df)):      
                       
        if  df.loc[i, "state"] == "out" :
            df.loc[i, "index_sc"] = (((df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"])+1)* df.loc[i-1,"index_sc"]
            df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)
            df.loc[i, "trade"] = (df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"]
            
        if  df.loc[i, "state"] == "enter" :        
            df.loc[i, "index_sc"] =  df.loc[i-1, "index_sc"]
            df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)
    dfindex = df
    return dfindex    #Columns : datetime	open	high	low	close	volume	state	index_sc	trade	index
    
def index_calculation (dfsignals, dfstoploss=None):
    if dfstoploss is not None:
        dfsignalsentoutclean = index_dataframe(dfsignals, dfstoploss)
        print (dfsignalsentoutclean)
    else:
        dfsignalsentoutclean = index_dataframe(dfsignals)
        display (dfsignalsentoutclean)
    dfindex = index_trade(dfsignalsentoutclean)
    return dfindex


In [484]:
dfindex = index_calculation (dfsignals, dfstoploss=None)
display (dfindex)


,datetime,open,high,low,close,volume,state
0,2024-11-11,80471.414062,89604.500000,80283.250000,88701.484375,1.179668e+11,enter
1,2024-12-09,101237.062500,101272.507812,94355.914062,97432.718750,1.106765e+11,out
2,2024-12-17,106030.687500,108268.445312,105291.734375,106140.601562,6.858936e+10,enter
3,2024-12-19,100070.687500,102748.148438,95587.679688,97490.953125,9.722166e+10,out
4,2025-07-10,111329.195312,116608.781250,110660.750000,115987.203125,9.591161e+10,enter
5,2025-07-31,117833.632812,118919.984375,115505.218750,115758.203125,6.937035e+10,out


,datetime,open,high,low,close,volume,state,index_sc,trade,index
0,2024-11-11,80471.414062,89604.500000,80283.250000,88701.484375,1.179668e+11,enter,100.000000,0.000000,99.700000
1,2024-12-09,101237.062500,101272.507812,94355.914062,97432.718750,1.106765e+11,out,109.843392,0.098434,109.513861
2,2024-12-17,106030.687500,108268.445312,105291.734375,106140.601562,6.858936e+10,enter,109.843392,0.000000,109.513861
3,2024-12-19,100070.687500,102748.148438,95587.679688,97490.953125,9.722166e+10,out,100.891994,-0.081492,100.589318
4,2025-07-10,111329.195312,116608.781250,110660.750000,115987.203125,9.591161e+10,enter,100.891994,0.000000,100.589318
5,2025-07-31,117833.632812,118919.984375,115505.218750,115758.203125,6.937035e+10,out,100.692797,-0.001974,100.390719


In [485]:
# Stop System Calculation , "stopsys" asignation into state column

def stop_drawdown (dfindex, drawmax):
    df = dfindex
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["index"] = pd.to_numeric(df["index"], errors="coerce")

    estado_corrigido = []
    pico_atual = df.loc[0, "index"]

    for i in range(len(df)):
        valor_index = df.loc[i, "index"]
        estado = df.loc[i, "state"]

        # Atualiza pico se houve recuperação
        if valor_index > pico_atual:
            pico_atual = valor_index

        # Calcula drawdown
        if pico_atual > 0:
            drawdown = (valor_index - pico_atual) / pico_atual
        else:
            drawdown = 0

        # Verifica se deve aplicar stopsys
        if estado == "out" and drawdown < (- drawmax):
            estado = "stopsys"
            pico_atual = valor_index  # reinicia ciclo a partir desse ponto

        estado_corrigido.append(estado)
    df["state"] = estado_corrigido
    dfindexdrawdown = df
    return dfindexdrawdown

dfindexdrawdown = stop_drawdown(dfindex, drawmax)

#display(dfindexdrawdown)

In [486]:
# METRICS
# creo dataframe para calculo de metricas
dfinputmetricas = dfindex[['datetime', 'state','index_sc', 'index','trade']]
#Metricas list inicialicion 

pd.set_option('display.max_rows', None)

display(dfinputmetricas, lsmetricas)

,datetime,state,index_sc,index,trade
0,2024-11-11,enter,100.000000,99.700000,0.000000
1,2024-12-09,out,109.843392,109.513861,0.098434
2,2024-12-17,enter,109.843392,109.513861,0.000000
3,2024-12-19,out,100.891994,100.589318,-0.081492
4,2025-07-10,enter,100.891994,100.589318,0.000000
5,2025-07-31,out,100.692797,100.390719,-0.001974


[K            15.000
 D             5.000
 smoth         5.000
 medM          4.000
 lowM          4.000
 stpl          0.020
 comission     0.003
 drawmax       0.200
 Name: 12, dtype: float64]

In [494]:
# Tir Total Metric
# Tir total calculation
def tir_total_anualizada(dfinputmetricas, lsmetricas):
    df = dfinputmetricas
    # Garante que datetime está no formato certo
    df['datetime'] = pd.to_datetime(df['datetime'])

    # Filtra enter e out
    df_enter = df[df['state'] == 'enter']
    df_out = df[df['state'] == 'out']

    # Verificação
    if df_enter.empty or df_out.empty:
        return None

    # Índice inicial e final
    idx_inicio = df_enter.iloc[0]['index']
    idx_fim = df_out.iloc[-1]['index']

    # Período completo entre primeira e última data do DataFrame
    dt_inicio_total = df['datetime'].min()
    dt_fim_total = df['datetime'].max()
    dias_total = (dt_fim_total - dt_inicio_total).days

    # Validação
    if dias_total <= 0 or idx_inicio == 0:
        return None

    # TIR anualizada com base no período total do df
    tirtotalanual = (idx_fim / idx_inicio) ** (365 / dias_total) - 1
    setirtotalanual = pd.Series({'tirtotalanual': tirtotalanual})
    lsmetricas.append (setirtotalanual)
    return  setirtotalanual , lsmetricas

setirtotalanual , lsmetricas = tir_total_anualizada (dfinputmetricas, lsmetricas)
#display (setirtotalanual, lsmetricas)

    

In [496]:
#%%timeit
#Tir Anuais Metrics
# dataframe  calculation
def tir_anuais_df (dfinputmetricas, dataini, datafim):

    # Exemplo do DataFrame original
    df = dfinputmetricas
    dataini = pd.to_datetime(dataini)
    datafim = pd.to_datetime(datafim)
    
    # Lista para novos registros
    novos_registros = []
    
    # Verifica se dataini deve ser adicionado
    if dataini < df.iloc[0]['datetime']:
        novos_registros.append({
            'datetime': dataini,
            'state': '',
            'index': df.iloc[0]['index']
        })
    
    # Verifica se datafim deve ser adicionado
    if datafim > df.iloc[-1]['datetime']:
        novos_registros.append({
            'datetime': datafim,
            'state': '',
            'index': df.iloc[-1]['index']
        })
    
    # Adiciona os registros e ordena
    df = pd.concat([pd.DataFrame(novos_registros), df], ignore_index=True)
    df = df.sort_values(by='datetime').reset_index(drop=True)
    
    # Determina os anos, excluindo o último ano
    ano_inicial = df['datetime'].min().year
    ano_final = (df['datetime'].max().year)
    
    # Gera os anos do intervalo EXCLUINDO o último ano
    anos_validos = range(ano_inicial, ano_final-1 )  # << ajuste aqui
    
    # Lista para os novos registros
    novos_registros = []
    
    for ano in anos_validos:
        fim_do_ano = pd.to_datetime(f'{ano}-12-31 23:59:59')
        df_antes = df[df['datetime'] < fim_do_ano]
        if not df_antes.empty:
            index_valor = df_antes.iloc[-1]['index']
            novos_registros.append({
                'datetime': fim_do_ano,
                'state': '',
                'index': index_valor
            })
    
    # Adiciona e organiza
    df = pd.concat([df, pd.DataFrame(novos_registros)], ignore_index=True)
    df = df.sort_values('datetime').reset_index(drop=True)
    
    # calcula o dataframe com as tir anuales ao fim do ano , com os anos incompletos anualizadas
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    #  consolidar por dia e manter o último registro
    df['date'] = df['datetime'].dt.date
    df_diario = df.sort_values('datetime').groupby('date', as_index=False).last()
    
    #  selecionar datas de fim de ano
    df_fim_ano = df_diario[
        (pd.to_datetime(df_diario['date']).dt.month == 12) &
        (pd.to_datetime(df_diario['date']).dt.day == 31)
    ].copy()
    
    #  calcular TIR entre pares de fim de ano
    resultados = []
    
    for i in range(1, len(df_fim_ano)):
        dt_inicio = pd.to_datetime(df_fim_ano.iloc[i - 1]['date'])
        dt_fim = pd.to_datetime(df_fim_ano.iloc[i]['date'])
        idx_inicio = df_fim_ano.iloc[i - 1]['index']
        idx_fim = df_fim_ano.iloc[i]['index']
        dias = (dt_fim - dt_inicio).days
    
        if dias > 0 and idx_inicio != 0:
            tir = (idx_fim / idx_inicio) ** (365 / dias) - 1
            resultados.append({
                'datetime': dt_fim,
                'tiranual': tir
            })
    
    #  adicionar último intervalo incompleto
    if not df_fim_ano.empty:
        dt_inicio = pd.to_datetime(df_fim_ano.iloc[-1]['date'])
        idx_inicio = df_fim_ano.iloc[-1]['index']
        dt_fim = pd.to_datetime(df_diario.iloc[-1]['date'])
        idx_fim = df_diario.iloc[-1]['index']
        dias = (dt_fim - dt_inicio).days
    
        if dias > 0 and idx_inicio != 0:
            tir = (idx_fim / idx_inicio) ** (365 / dias) - 1
            resultados.append({
                'datetime': dt_fim,
                'tiranual': tir
            })

    # criar DataFrame final
    dftiranual = pd.DataFrame(resultados)
    return dftiranual

dftiranual = tir_anuais_df (dfinputmetricas, dataini, datafim)

#estatistic calculation
def tir_anuais_estat(dftiranual, lsmetricas):
    tir = dftiranual['tiranual'].dropna()

    estatisticas = {
        'tiranualquant': tir.count(),
        'tiranualfirst': round(tir.iloc[0], 6),
        'tiranualmedia': round(tir.mean(), 6),
        'tiranualmax': round(tir.max(), 6),
        'tiranualmin': round(tir.min(), 6),
        'tiranualstd': round(tir.std(), 6)
    }
    setiranuaisestats = pd.Series(estatisticas)
    lsmetricas.append (setiranuaisestats)
    return setiranuaisestats , lsmetricas

setiranuaisestats , lsmetricas = tir_anuais_estat(dftiranual, lsmetricas)
#display (dftiranual, lsmetricas)


In [498]:
# TRades Metrics
# trades estatistics calculation
def trades_estatisticas(dfinputmetricas, lsmetricas):
    df = dfinputmetricas.copy()
    trades = df["trade"].dropna()

    positivos = trades[trades > 0]
    negativos = trades[trades < 0]

    # Porcentagem de positivos
    porcentagem_pos = (len(positivos) / len(trades)) if len(trades) > 0 else 0

    ditradesestat = {
        "tradestot": len(trades),
        'tradefirst': round(trades.iloc[1], 6),
        "tradespositpor": round(porcentagem_pos, 6),
        "tradesposmedia": round(positivos.mean(), 6) if not positivos.empty else None,
        "tradesposstd": round(positivos.std(), 6) if not positivos.empty else None,
        "tradesposmax": round(positivos.max(), 6) if not positivos.empty else None,
        "tradesposmin": round(positivos.min(), 6) if not positivos.empty else None
    }

    setradesestats = pd.Series(ditradesestat)
    lsmetricas.append (setradesestats)
    return setradesestats , lsmetricas

setradesestats, lsmetricas = trades_estatisticas(dfinputmetricas, lsmetricas)
#display (lsmetricas)

In [500]:
#Drawdown Metrics

#Dataframe Drawdown Calculation
def drawdowns_df (dfinputmetricas):
   
    df = dfinputmetricas
    df["datetime"] = pd.to_datetime(df["datetime"])
    serie = df["index"].dropna().reset_index(drop=True)
    datas = df["datetime"].reset_index(drop=True)

    drawdowns = []

    pico_idx = 0
    pico = serie[0]
    vale_idx = None
    valor_vale = None
    max_dd = 0

    for i in range(1, len(serie)):
        if serie[i] > pico:
            # Se recuperou acima do último pico: salvar ciclo anterior
            if vale_idx is not None and max_dd < 0:
                drawdowns.append({
                    "Data Pico": datas[pico_idx],
                    "Valor Pico": pico,
                    "Data Vale": datas[vale_idx],
                    "Valor Vale": valor_vale,
                    "Drawdown (%)": round(max_dd * 100, 2)
                })

            # Novo pico inicia novo ciclo
            pico = serie[i]
            pico_idx = i
            vale_idx = None
            max_dd = 0
        else:
            dd = (serie[i] - pico) / pico
            if dd < max_dd:
                max_dd = dd
                vale_idx = i
                valor_vale = serie[i]

    # Salva último ciclo, se aplicável
    if vale_idx is not None and max_dd < 0:
        drawdowns.append({
            "Data Pico": datas[pico_idx],
            "Valor Pico": pico,
            "Data Vale": datas[vale_idx],
            "Valor Vale": valor_vale,
            "Drawdown (%)": round(max_dd * 100, 2)
        })

    # Retorna os top N
    df_resultado = pd.DataFrame(drawdowns)
    return df_resultado.sort_values("Drawdown (%)").reset_index(drop=True)  

dfdrawdowns = drawdowns_df(dfinputmetricas)

# Drawdowns statictics calculation
def drawdowns_estat(dfdrawdowns , lsmetricas):
    dd = dfdrawdowns['Drawdown (%)'].dropna()  # Filtra nulos, se houver

    estatisticas = {
        'drawdfirst': round(dd.iloc[0], 6),
        'drawdtot': dd.count(),
        'drawdmedia': round(dd.mean(), 2),
        'drawdmaximo': round(dd.max(), 2),
        'drawdminimo': round(dd.min(), 2),
        'drawdstd': round(dd.std(), 2)
    }
    sedrawdownsestats = pd.Series(estatisticas)
    lsmetricas.append (sedrawdownsestats)
    return sedrawdownsestats , lsmetricas

sedrawdownsestats , lsmetricas = drawdowns_estat(dfdrawdowns, lsmetricas)
#display(lsmetricas)

In [502]:
import pandas as pd

def dias_out_df(dfmetricas):
    df = dfmetricas.copy()
    coluna = "index_sc"

    # Converter e filtrar linhas sem valor em index_sc ou datetime inválido
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    df = df[df[coluna].notna() & df["datetime"].notna()].reset_index(drop=True)

    # Identificar grupos estáticos
    variacao = df[coluna].diff()
    grupos = (variacao != 0).cumsum()
    periodos = []

    for _, grupo in df.groupby(grupos):
        if len(grupo) > 1 and grupo[coluna].nunique() == 1:
            dias = (grupo["datetime"].iloc[-1] - grupo["datetime"].iloc[0]).days
            periodos.append({
                "Data Início": grupo["datetime"].iloc[0],
                "Data Fim":   grupo["datetime"].iloc[-1],
                "difdias":    dias,
                "Valor index": grupo[coluna].iloc[0]
            })

    dfdiasout = pd.DataFrame(periodos)

    # Se não houver a coluna difdias ou ela estiver vazia, retorne vazio
    if "difdias" not in dfdiasout.columns or dfdiasout.empty:
        return pd.DataFrame(columns=["Data Início", "Data Fim", "difdias", "Valor index"])

    # Filtragem direta sem query
    dfdiasout = dfdiasout[dfdiasout["difdias"] != 0].reset_index(drop=True)
    return dfdiasout


def dias_out_estats(dfdiasout, lsmetricas):
    # Se estiver vazio, retorna Series zerada ou NaNs
    if dfdiasout.empty:
        estat = {
            'diasoutfirst': None,
            'diasouttot':   0,
            'diasoutmedia': None,
            'diasoutmax':   None,
            'diasoutmin':   None,
            'diasoutstd':   None
        }
    else:
        dias = dfdiasout["difdias"]
        estat = {
            'diasoutfirst': round(dias.iloc[0], 6),
            'diasouttot':   dias.sum(),
            'diasoutmedia': round(dias.mean(), 2),
            'diasoutmax':   dias.max(),
            'diasoutmin':   dias.min(),
            'diasoutstd':   round(dias.std(), 2)
        }

    series_estat = pd.Series(estat)
    lsmetricas.append(series_estat)
    return series_estat, lsmetricas

#Dias Out Metrics  erro quando dias out = 0

# Dataframe calculation
def dias_out_df (dfmetricas):    
    df = dfmetricas
    coluna="index_sc"
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df[df[coluna].notna()].reset_index(drop=True)

    variacao = df[coluna].diff()
    grupos = (variacao != 0).cumsum()

    agrupado = df.groupby(grupos)
    periodos_estaticos = []

    for _, grupo in agrupado:
        if len(grupo) > 1 and grupo[coluna].nunique() == 1:
            duracao_dias = (grupo["datetime"].iloc[-1] - grupo["datetime"].iloc[0]).days
            periodos_estaticos.append({                
                "Data Início": grupo["datetime"].iloc[0],
                "Data Fim": grupo["datetime"].iloc[-1],
                "difdias": duracao_dias,
                "Valor index": grupo[coluna].iloc[0]
            })

    dfdiasout = pd.DataFrame(periodos_estaticos)
    dfdiasout = dfdiasout.query("difdias != 0").copy()
    dfdiasout = dfdiasout.reset_index(drop=True)
    
    return dfdiasout
    
dfdiasout = dias_out_df (dfinputmetricas)
display (dfdiasout)

# Diasout statistic calculation and lsmetricas agregation
def dias_out_estats(dfdiasout, lsmetricas) :
    dias = dfdiasout['difdias'].dropna()  # Remove valores nulos, se houver

    estatisticas = {
        'diasoutfirst': round(dias.iloc[0], 6),
        'diasouttot': dias.sum(),
        'diasoutmedia': round(dias.mean(), 2),
        'diasoutmax': dias.max(),
        'diasoutmin': dias.min(),
        'diasoutstd': round(dias.std(), 2)
    }
    sediasoutestats = pd.Series(estatisticas)
    lsmetricas.append(sediasoutestats)    
    return sediasoutestats, lsmetricas

sediasoutestats, lsmetricas = dias_out_estats(dfdiasout, lsmetricas)
#display(lsmetricas)

In [505]:
#Metrica StopSys

# Dataframe calculation
def stopsys_df (dfindexdrawdown) : 
    df = dfindexdrawdown
    # Garante que a coluna 'datetime' esteja no formato correto
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    # Filtra os registros onde state == 'stopsys'
    dfstopsys = df[df['state'] == 'stopsys'][['datetime', 'state', 'index']].copy()
    
    #  Ordena por datetime
    dfstopsys = dfstopsys.sort_values('datetime').reset_index(drop=True)
    return dfstopsys
    
dfstopsys = stopsys_df (dfindexdrawdown)

# statistic dataframe based calculation and lsmetricas agregation
def stopsys_estat(dfstopsys, lsmetricas):
    df = dfstopsys.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)

    index_values = df['index'].dropna()

    if index_values.empty:
        estatisticas = {
            'stopsysfirst': 0.0,
            'stopsysquant': 0,
            'stopsysmedia': 0.0,
            'stopsysmaximo': 0.0,
            'stopsysminimo': 0.0,
            'stopsystd': 0.0
        }
    else:
        estatisticas = {
            'stopsysfirst': round(index_values.iloc[0], 6),
            'stopsysquant': index_values.count(),
            'stopsysmedia': round(index_values.mean(), 6),
            'stopsysmaximo': round(index_values.max(), 6),
            'stopsysminimo': round(index_values.min(), 6),
            'stopsystd': round(index_values.std(), 6)
        }

    sestopsysestats = pd.Series(estatisticas)
    lsmetricas.append(sestopsysestats)
    return sestopsysestats, lsmetricas
sestopsysestats, lsmetricas = stopsys_estat(dfstopsys, lsmetricas)
#display (lsmetricas )

In [507]:


def atualizar_df_metricas(dfmetricas, lsmetricas):
    """
    Adiciona uma linha ao DataFrame dfmetricas com os valores de lsmetricas.
    Se dfmetricas for None, cria o DataFrame com a estrutura das métricas.

    Parâmetros:
    - dfmetricas: pd.DataFrame ou None
    - lsmetricas: list de pd.Series

    Retorna:
    - pd.DataFrame atualizado
    """

    linha = pd.concat(lsmetricas)  # Une todas as Series em uma só

    if dfmetricas is None:
        # Cria o DataFrame com uma única linha
        dfmetricas = pd.DataFrame([linha.values], columns=linha.index)
    else:
        # Adiciona nova linha ao DataFrame existente
        dfmetricas.loc[len(dfmetricas)] = linha.values

    return dfmetricas

dfmetricas = atualizar_df_metricas(dfmetricas, lsmetricas)

In [509]:
display (dfmetricas)

,K,D,smoth,medM,lowM,stpl,comission,drawmax,tirtotalanual,tiranualquant,...,drawdmedia,drawdmaximo,drawdminimo,drawdstd,stopsysfirst,stopsysquant,stopsysmedia,stopsysmaximo,stopsysminimo,stopsystd
0,15.0,15.0,5.0,4.0,4.0,0.02,0.003,0.2,-0.716462,1.0,...,-2.05,-2.05,-2.05,NaN,0.0,0.0,0.0,0.0,0.0,0.0
1,5.0,5.0,5.0,4.0,4.0,0.02,0.003,0.2,0.496272,1.0,...,-5.30,-2.07,-8.52,4.56,0.0,0.0,0.0,0.0,0.0,0.0
2,5.0,15.0,5.0,4.0,4.0,0.02,0.003,0.2,0.033488,1.0,...,-8.74,-8.74,-8.74,NaN,0.0,0.0,0.0,0.0,0.0,0.0
3,10.0,10.0,5.0,4.0,4.0,0.02,0.003,0.2,0.021222,1.0,...,-8.09,-8.09,-8.09,NaN,0.0,0.0,0.0,0.0,0.0,0.0
4,15.0,5.0,5.0,4.0,4.0,0.02,0.003,0.2,0.009665,1.0,...,-8.33,-8.33,-8.33,NaN,0.0,0.0,0.0,0.0,0.0,0.0


END LOOP

In [329]:
dfmetricas.insert(
    loc=0,  # insere como primeira coluna
    column="id_backtest",
    value=[srbacktest["id_backtest"]] * len(dfmetricas)
)
display(dfmetricas)

,id_backtest,K,D,smoth,medM,lowM,stpl,comission,drawmax,tirtotalanual,...,diasoutmedia,diasoutmax,diasoutmin,diasoutstd,stopsysfirst,stopsysquant,stopsysmedia,stopsysmaximo,stopsysminimo,stopsystd
0,1,16.0,12.0,6.0,5.0,10.0,0.04,0.005,-0.05,-0.008228,...,25.94,103.0,1.0,30.96,92.631437,18.0,80.080879,105.007263,59.073016,13.316339
